# Data Processing — chunk the corpus (BM25 scope)

This is the **ingestion** notebook: the "build it once" stage. For the BM25
baseline it only needs to:

1. read the corpus + queries parquet from S3,
2. chunk the corpus into fixed word windows,
3. write the chunks back to S3.

No embedding / vector DB yet — BM25 is lexical, so there's nothing to embed.
When we add **dense** retrieval, this notebook grows an embed step + a FAISS
index build, writing the index to `s3://.../indexes/`. That's when the
"vector database" part appears.

**Prereqs in Studio:** upload the `rag_eval/` package folder and `configs/`
so they sit next to this notebook. This notebook imports the existing harness
rather than reimplementing it.

## 1. Imports + session

Inside Studio you don't pass an AWS profile — boto3 automatically uses the
domain's **execution role** (the one we attached S3 + Bedrock access to). The
import-path print confirms the uploaded `rag_eval/` is visible.

In [ ]:
import sys, io, os
sys.path.insert(0, ".")  # rag_eval/ sits next to this notebook

import boto3
import pandas as pd

import rag_eval
print("rag_eval imported from:", os.path.dirname(rag_eval.__file__))

REGION = "us-east-1"
BUCKET = "rag-arch-eval-data-875699230934"
DATA_PREFIX = "data"       # input:  corpus.parquet, queries.parquet
CHUNKS_PREFIX = "chunks"   # output: chunks.parquet

s3 = boto3.client("s3", region_name=REGION)
print("caller:", boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"])

## 2. Read the corpus + queries from S3

We read the parquet straight from S3 into pandas. `corpus` is 609 news
articles (title is the unique doc_id); `queries` is 2,556 multi-hop questions.

In [ ]:
def read_parquet_s3(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))

corpus = read_parquet_s3(f"{DATA_PREFIX}/corpus.parquet")
queries = read_parquet_s3(f"{DATA_PREFIX}/queries.parquet")
print("corpus rows:", len(corpus), "| unique titles:", corpus['title'].nunique())
print("queries rows:", len(queries))
corpus.head(2)

## 3. Build `Doc` objects and chunk

We reuse `rag_eval.data.Doc` and `rag_eval.chunking.chunk_words` — the same
code the local harness and tests use. 256-word windows, 32-word overlap,
title prepended (matches `configs/bm25.yaml`). Each chunk keeps its parent
`doc_id` so retrieval is scored at the document level.

In [ ]:
from rag_eval.data import Doc
from rag_eval.chunking import chunk_words

docs = [
    Doc(r.title, r.title, r.body, r.source, str(r.published_at), r.category, r.url)
    for r in corpus.itertuples()
]

chunks = chunk_words(docs, size=256, overlap=32, prepend_title=True)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
print("example chunk_id:", chunks[0].chunk_id)
print("example text (first 200 chars):\n", chunks[0].text[:200])

## 4. Write the chunks back to S3

We persist chunks as parquet so the eval notebook can load them directly
without re-chunking. Columns: `chunk_id`, `doc_id`, `text`.

In [ ]:
chunks_df = pd.DataFrame(
    [{"chunk_id": c.chunk_id, "doc_id": c.doc_id, "text": c.text} for c in chunks]
)

buf = io.BytesIO()
chunks_df.to_parquet(buf, index=False)
buf.seek(0)
out_key = f"{CHUNKS_PREFIX}/chunks.parquet"
s3.put_object(Bucket=BUCKET, Key=out_key, Body=buf.getvalue())
print(f"wrote s3://{BUCKET}/{out_key}  ({len(chunks_df)} chunks, {buf.getbuffer().nbytes} bytes)")

## 5. Verify the round-trip

Read the chunks back and confirm the count + schema match what we wrote.

In [ ]:
check = read_parquet_s3(out_key)
print("read back rows:", len(check))
print("columns:", list(check.columns))
assert len(check) == len(chunks_df), "chunk count mismatch"
print("OK — chunks persisted. (BM25 needs nothing more.)")

---
# Dense ingestion (Step 6)

BM25 was done at the chunk-write step above. **Dense** retrieval needs more:
we embed every chunk with Amazon Titan, build a FAISS index from those
vectors, and persist the index to S3. This is the "vector database" stage.

Two artifacts go to `s3://.../indexes/`:
- `faiss.index` — the serialized FAISS index (vectors only, by position).
- `id_map.parquet` — row-position -> (chunk_id, doc_id). FAISS stores no IDs,
  so this sidecar maps a search hit back to a document. **Order must match**
  the order vectors were added to the index.

Cost/time note: embedding ~a few thousand chunks is a few thousand Titan
calls — cents, but a minute or two of sequential calls, so the cell isn't
hung, just working. `TitanEmbedder` caches to disk (EFS in Studio), so
re-runs are instant.

## D1. Install deps + embed the chunks

`faiss-cpu` for the index, `boto3` is already present. The `TitanEmbedder`
from the uploaded `rag_eval` package handles the Bedrock calls + caching.

In [ ]:
%pip install -q faiss-cpu==1.15.1
import numpy as np
import faiss
from rag_eval.bedrock import TitanEmbedder

# In Studio, no profile — the embedder's boto3 client uses the exec role.
embedder = TitanEmbedder(model_id="amazon.titan-embed-text-v2:0", region=REGION)

texts = [c.text for c in chunks]
print(f"embedding {len(texts)} chunks with Titan (cached on repeat runs)...")
vecs = embedder.embed(texts)            # (n, 1024) float32
print("embeddings shape:", vecs.shape)

## D2. Build the FAISS index

Normalize vectors so inner product == cosine similarity, then add them to an
exact `IndexFlatIP`. Exact search = perfect recall; fine at this corpus size.
The id_map preserves the row order we add vectors in.

In [ ]:
def normalize(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); n[n == 0] = 1.0
    return m / n

vecs_n = normalize(vecs.astype(np.float32))
index = faiss.IndexFlatIP(vecs_n.shape[1])
index.add(vecs_n)
print("index size:", index.ntotal, "dim:", index.d)

id_map = pd.DataFrame(
    {"row": range(len(chunks)),
     "chunk_id": [c.chunk_id for c in chunks],
     "doc_id": [c.doc_id for c in chunks]}
)

## D3. Write the index + id_map to S3

`faiss.write_index` serializes to a local file; we upload that bytes blob and
the id_map parquet to `s3://.../indexes/`.

In [ ]:
import tempfile, os
INDEXES_PREFIX = "indexes"

with tempfile.TemporaryDirectory() as d:
    local = os.path.join(d, "faiss.index")
    faiss.write_index(index, local)
    s3.upload_file(local, BUCKET, f"{INDEXES_PREFIX}/faiss.index")
print(f"wrote s3://{BUCKET}/{INDEXES_PREFIX}/faiss.index")

buf = io.BytesIO(); id_map.to_parquet(buf, index=False); buf.seek(0)
s3.put_object(Bucket=BUCKET, Key=f"{INDEXES_PREFIX}/id_map.parquet", Body=buf.getvalue())
print(f"wrote s3://{BUCKET}/{INDEXES_PREFIX}/id_map.parquet  ({len(id_map)} rows)")
print("OK — dense index persisted. Next: open dense_eval.ipynb")